## ScrapingAir4thai.py

In [1]:
import requests
import pandas as pd
from datetime import datetime

url = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php?region=1"
data = requests.get(url).json()

records = []

for station in data["stations"]:
    info = station.get("AQILast", {})
    
    comment = f"สถานี: {station.get('nameTH', '')} | "
    comment += f"PM2.5: {info.get('PM2.5', {}).get('value', '-')}"
    comment += f" (AQI {info.get('PM2.5', {}).get('aqi', '-')}) | "
    comment += f"PM10: {info.get('PM10', {}).get('value', '-')} | "
    comment += f"CO: {info.get('CO', {}).get('value', '-')} | "
    comment += f"O3: {info.get('O3', {}).get('value', '-')} | "
    comment += f"NO2: {info.get('NO2', {}).get('value', '-')} | "
    comment += f"SO2: {info.get('SO2', {}).get('value', '-')} | "
    comment += f"เวลา: {info.get('date', '')} {info.get('time', '')}"

    records.append({
        "type": "{PM2.5}",
        "organization": "Air4Thai",
        "comment": comment,
        "photo": "",
        "photo_after": "",
        "coords": f"{station.get('lat', '')},{station.get('long', '')}",
        "address": station.get("areaTH", ""),
        "subdistrict": "",
        "district": "",
        "province": "กรุงเทพมหานคร",
        "timestamp": datetime.now().isoformat(),
        "state": "open",
        "star": 0,
        "count_reopen": 0,
        "last_activity": datetime.now().isoformat()
    })

df_out = pd.DataFrame(records)

df_out["ticket_id"] = ["EXT-%05d" % i for i in range(1, len(df_out)+1)]

cols = ["ticket_id"] + [col for col in df_out.columns if col != "ticket_id"]
df_out = df_out[cols]

df_out.to_csv("../data_raw/external_raw/pm25_api_full.csv", index=False, encoding="utf-8-sig")
print("✅ บันทึกข้อมูลสำเร็จ:", len(df_out), "รายการ")


✅ บันทึกข้อมูลสำเร็จ: 90 รายการ


## ScrapingFlood_risk_gistda.py

In [2]:
import requests
import pandas as pd
from datetime import datetime

url = "https://api-gateway.gistda.or.th/api/2.0/resources/features/flood/30days"
headers = {
    "accept": "application/json",
    "API-Key": "EFH7ZXZ0riZUP9oMgiRyC6x8UESdYK4CNOffGjF9x4d6lfrxq8V4gNUUIgWxnCJe"
}
params = {
    "bbox": "100.28,13.52,100.95,14.2",
    "limit": 1000,
    "offset": 0
}

response = requests.get(url, headers=headers, params=params)
data = response.json()

records = []
for feature in data.get("features", []):
    props = feature.get("properties", {})
    coords = feature.get("geometry", {}).get("coordinates", [])

    # หาค่ากลางของพิกัด polygon
    try:
        lat = coords[0][0][0][1]
        lon = coords[0][0][0][0]
        coord_str = f"{lat},{lon}"
    except Exception:
        coord_str = ""

    records.append({
        "type": "{น้ำท่วม}",
        "organization": "GISTDA",
        "comment": props.get("LabelTH", "น้ำท่วม") + " | พท. (ตร.ม.): " + str(props.get("shape_area", "")),
        "photo": "",
        "photo_after": "",
        "coords": coord_str,
        "address": "",
        "subdistrict": "",
        "district": "",
        "province": "กรุงเทพมหานคร",
        "timestamp": props.get("_updatedAt", datetime.now().isoformat()),
        "state": "",  
        "star": 0,
        "count_reopen": 0,
        "last_activity": datetime.now().isoformat()
    })

df_out = pd.DataFrame(records)

df_out["ticket_id"] = ["EXT-%05d" % (i + 91) for i in range(len(df_out))]

cols = ["ticket_id"] + [col for col in df_out.columns if col != "ticket_id"]
df_out = df_out[cols]

df_out.to_csv("../data_raw/external_raw/flood_30days_gistda_formatted.csv", index=False, encoding="utf-8-sig")
print("✅ บันทึกข้อมูลในรูปแบบ df_out สำเร็จ:", len(df_out), "รายการ")




✅ บันทึกข้อมูลในรูปแบบ df_out สำเร็จ: 1000 รายการ


## ScrapingFloodrisk.py

In [3]:
import pandas as pd
from datetime import datetime

csv_url = "https://data.bangkok.go.th/dataset/3fcbe9f7-d2b0-4442-b110-4a5620ebdce3/resource/8f1102d5-52a2-4494-9131-403e4f87a242/download/flood_risk.csv"

df_raw = pd.read_csv(csv_url)

# สร้าง DataFrame ตาม schema ที่ต้องการ

df_out = pd.DataFrame({
    "type": "{น้ำท่วม}",
    "organization": "Bangkok Flood Report",
    "comment": df_raw["name"] + " | รายละเอียด: " + df_raw["detail"].fillna(""),
    "photo": "",
    "photo_after": "",
    "coords": df_raw["y"].astype(str) + "," + df_raw["x"].astype(str),
    "address": df_raw["name"],
    "subdistrict": "",  # ถ้ายังไม่มีข้อมูลแขวง ให้เว้นไว้
    "district": df_raw["district"],
    "province": "กรุงเทพมหานคร",
    "timestamp": datetime.now().isoformat(),  # หรือใช้คอลัมน์เวลา (ถ้ามี)
    "state": df_raw["status_detail"].fillna(""),  
    "star": 0,
    "count_reopen": 0,
    "last_activity": datetime.now().isoformat()
})

df_out["ticket_id"] = ["EXT-%05d" % (i + 1091) for i in range(len(df_out))]

cols = ["ticket_id"] + [col for col in df_out.columns if col != "ticket_id"]
df_out = df_out[cols]

df_out.to_csv("../data_raw/external_raw/flood_risk!!!.csv", index=False, encoding="utf-8-sig")
print("✅ แปลงข้อมูลสำเร็จ:", len(df_out), "รายการ")


✅ แปลงข้อมูลสำเร็จ: 737 รายการ


## ScrapingRiskpath.py

In [4]:
import requests
import pandas as pd
from datetime import datetime

# โหลด JSON จากไฟล์หรือ URL
url = "https://data.go.th/dataset/d37c09dc-6939-492c-b7bf-6e15d3597998/resource/6dccd462-9d84-4f53-b8c2-7bfcda7e1d7c/download/risk.json"
data = requests.get(url).json()

# ดึง feature list
features = data["features"]

# เตรียมข้อมูลให้อยู่ในรูป list of dict
rows = []
for feature in features:
    prop = feature["properties"]
    coords = feature["geometry"]["coordinates"]
    
    rows.append({
        "type": "{ที่ตั้งจุดเสี่ยงภัยสะพานลอยและป้ายรถโดยสารประจำทาง}", 
        "organization": "กรมป้องกันและบรรเทาสาธารณภัย",
        "comment": prop["location"],
        "photo": "",
        "photo_after": "",
        "coords": f"{coords[1]},{coords[0]}",  # NOTE: Y,X
        "address": prop["location"],
        "subdistrict": "",  # ไม่ระบุในข้อมูลต้นทาง
        "district": "",     # ไม่ระบุเช่นกัน
        "province": "กรุงเทพมหานคร",  # จาก dcode พอเดาได้ว่าอยู่ใน กทม.
        "timestamp": datetime.now().isoformat(),
        "state": "",  # ไม่มีข้อมูลเพิ่มเติมใน field นี้
        "star": 0,
        "count_reopen": 0,
        "last_activity": datetime.now().isoformat()
    })

# สร้าง DataFrame และบันทึกเป็น CSV
df_out = pd.DataFrame(rows)

df_out["ticket_id"] = ["EXT-%05d" % (i + 3828) for i in range(len(df_out))]

cols = ["ticket_id"] + [col for col in df_out.columns if col != "ticket_id"]
df_out = df_out[cols]

df_out.to_csv("../data_raw/external_raw/risk_path.csv", index=False, encoding="utf-8-sig")

print("✅ บันทึกไฟล์ risk_points_cleaned.csv เรียบร้อย:", len(df_out), "รายการ")


✅ บันทึกไฟล์ risk_points_cleaned.csv เรียบร้อย: 483 รายการ


## merge_external

In [5]:
import pandas as pd
import glob

# เลือกไฟล์ทั้งหมดในโฟลเดอร์ (นามสกุล .csv)
csv_files = glob.glob("../data_raw/external_raw/*.csv")

# อ่านและรวมไฟล์ทั้งหมด
df_list = []
for file in csv_files:
    df_list.append(pd.read_csv(file))

# ต่อไฟล์แบบต่อตูด
merged_df = pd.concat(df_list, ignore_index=True)

# บันทึกไฟล์รวม
merged_df.to_csv("../data_raw/all_external.csv", index=False, encoding="utf-8-sig")



## clean_external

In [6]:
import pandas as pd

# อ่านไฟล์ต้นฉบับ
df = pd.read_csv("../data_raw/all_external.csv")

# กรองแถวที่ district ไม่เป็น NaN
df_clean = df.dropna(subset=["district"])

# เซฟออกเป็นไฟล์ใหม่
df_out.to_csv("../data_clean/all_external_clean.csv", index=False, encoding="utf-8-sig")
